解析pdf文件，获得脏数据，使用Mineru镜像

In [ ]:
import os 
from pathlib import Path 
import logging
from datetime import datetime
import subprocess
import shutil
import time

log_dir = Path( Path.cwd().parents[0] , "logging")
if not log_dir.exists():
    log_dir.mkdir(parents=True, exist_ok=True)
log_dir = Path( Path.cwd().parents[0] , "logging")
if not log_dir.exists():
    log_dir.mkdir(parents=True, exist_ok=True)
log_file = log_dir / f"pdf_parser_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file, encoding='utf-8'),
        logging.StreamHandler()
    ]
)
print(f"📄 log保存位置: {log_file}")
print(f"📁 log目录: {log_dir}")
p = Path( Path.cwd().parents[0], 'data' , 'raw' , '示例数据','附件2：财务报告' )
dirs_names  = [Path(p, name) for name in os.listdir(p)]
all_files = []
for dir in dirs_names:
    files = [Path(dir, name) for name in os.listdir(dir)]
    all_files.extend(files)
all_files

In [ ]:

# ===== 配置 =====
PDF_ROOT = Path("/root/TAIDIBEI_B/data/raw/示例数据/附件2：财务报告")
OUTPUT_ROOT = Path("/root/TAIDIBEI_B/parsed_results")
LOG_DIR = Path("/root/TAIDIBEI_B/logs")
# ================

# 创建目录
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# 配置日志
log_file = LOG_DIR / f"processing_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file, encoding='utf-8'),
        logging.StreamHandler()
    ]
)

def process_pdf(pdf_path):
    """处理单个PDF并自动移动文件"""
    
    # 提取信息
    company = pdf_path.parent.name  # reports-上交所 或 reports-深交所
    pdf_name = pdf_path.stem  # 不带后缀的文件名
    
    # 创建目标目录
    target_dir = OUTPUT_ROOT / company
    images_dir = target_dir / "images"


    target_dir.mkdir(parents=True, exist_ok=True)
    images_dir.mkdir(parents=True, exist_ok=True)
    
    logging.info(f"处理: {pdf_path.name}")
    logging.info(f"目标: {target_dir}")
    
    # 运行magic-pdf命令
    cmd = ["magic-pdf", "pdf-command", "--pdf", str(pdf_path), "--method", "auto"]
    
    try:
        # 执行命令
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
        
        if result.returncode == 0:
            # 查找临时目录中的结果
            tmp_dir = Path(f"/tmp/magic-pdf/{pdf_name}/auto")
            
            if tmp_dir.exists():
                # 移动markdown文件
                md_files = list(tmp_dir.glob("*.md"))
                for md in md_files:
                    shutil.move(str(md), str(target_dir / md.name))
                    logging.info(f"  📄 移动: {md.name}")
                
                # 移动json文件
                json_files = list(tmp_dir.glob("*.json"))
                for json in json_files:
                    shutil.move(str(json), str(target_dir / json.name))
                    logging.info(f"  📊 移动: {json.name}")
                
                # 移动图片
                img_dir = tmp_dir / "images"
                if img_dir.exists():
                    jpg_count = 0 
                    for img in img_dir.glob("*.jpg"):
                        shutil.move(str(img), str(images_dir / img.name))
                        jpg_count += 1
                    logging.info(f"  🖼️  移动图片: {jpg_count}张")
                    img_dir.rmdir()  # 删除空目录
                
                if tmp_dir.parent.exists():
                    shutil.rmtree(tmp_dir.parent) 
                    logging.info(f"  🗑️ 清理临时目录: {tmp_dir.parent}")
                
                logging.info(f"  ✅ 完成: {pdf_name}")
                return True
            else:
                logging.warning(f"  ⚠️ 未找到临时目录: {tmp_dir}")
                return False
        else:
            logging.error(f"  ❌ 处理失败: {pdf_path.name}")
            if result.stderr:
                logging.error(f"     错误: {result.stderr[:200]}")
            return False
            
    except subprocess.TimeoutExpired:
        logging.error(f"  ⏰ 超时: {pdf_path.name}")
        return False
    except Exception as e:
        logging.error(f"  💥 异常: {pdf_path.name}, {str(e)}")
        return False

def main():
    # 查找所有PDF
    all_pdfs = list(PDF_ROOT.rglob("*.pdf"))
    logging.info(f"="*60)
    logging.info(f"🚀 开始处理，共 {len(all_pdfs)} 个PDF文件")
    logging.info(f"📁 输出目录: {OUTPUT_ROOT}")
    logging.info(f"📄 日志文件: {log_file}")
    logging.info(f"="*60)
    
    success = 0
    failed = 0
    start_time = time.time()
    
    for i, pdf in enumerate(all_pdfs, 1):
        logging.info(f"\n[{i}/{len(all_pdfs)}] 进度: {i/len(all_pdfs)*100:.1f}%")
        
        if process_pdf(pdf):
            success += 1
        else:
            failed += 1
        
        # 每5个文件休息一下
        if i % 5 == 0 and i < len(all_pdfs):
            logging.info(f"⏸️  已处理 {i} 个，休息3秒...")
            time.sleep(3)
    
    # 统计
    elapsed = time.time() - start_time
    hours = int(elapsed // 3600)
    minutes = int((elapsed % 3600) // 60)
    seconds = int(elapsed % 60)
    
    logging.info("\n" + "="*60)
    logging.info(f"✅ 全部处理完成！")
    logging.info(f"📊 统计:")
    logging.info(f"   总文件: {len(all_pdfs)}")
    logging.info(f"   成功: {success}")
    logging.info(f"   失败: {failed}")
    logging.info(f"   用时: {hours}小时{minutes}分钟{seconds}秒")
    logging.info(f"📁 结果目录: {OUTPUT_ROOT}")
    logging.info(f"📄 日志文件: {log_file}")
    logging.info("="*60)

if __name__ == "__main__":
    main()

解析markdown数据，使用大模型图生文大模型进行图片解析，调用？？模型。
一个新的虚拟环境
conda create -n py10 python=3.10
conda activate py10
pip install numpy pandas matplotlib scikit-learn
pip install ipykernel jupyter

python -m ipykernel install --user --name=py10 --display-name="py10"
Installed kernelspec py10 in C:\Users\86193\AppData\Roaming\jupyter\kernels\py10

pip install python-dotenv #配置全局环境
pip install zai-sdk

In [ ]:
#传入 Base64 图片
from zai import ZhipuAiClient
from dotenv import load_dotenv
import base64
import os
load_dotenv() 
client = ZhipuAiClient(api_key=os.getenv("ZHIPU_API_KEY"))  # 填写您自己的APIKey

img_path = "E:\\github\\TAIDIBEI_B\\parsed_results\\reports-上交所\\images\\0a7ccf0742874e9a67f8048602eb1607f392024e52a3ecf2ed8eccbdbcffa0c4.jpg"
with open(img_path, "rb") as img_file:
    img_base = base64.b64encode(img_file.read()).decode("utf-8")

response = client.chat.completions.create(
    model="glm-4.6v",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": img_base
                    }
                },
                {
                    "type": "text",
                    "text": "请描述这个图片"
                }
            ]
        }
    ]
)
print(response.choices[0].message)

APIRequestFailedError: Error code: 400, with error text {"error":{"code":"1214","message":"OCR仅支持PDF、JPG、PNG、JPEG格式；文件大小限制：图片≤10MB、PDF≤50MB；PDF最大100页"}}

In [3]:
print( response.choices[0].message.content )


这是一张关于其他应收款期末余额及相关信息的表格，单位为元，币种为人民币。表格包含以下列：**单位名称**、**期末余额**、**占其他应收款期末余额合计数的比例(%)**、**款项的性质**、**账龄**、**坏账准备期末余额**。各行数据如下：  

1. **西部投资集团有限公司**：期末余额153,280,700.00元，占比65.97%；款项性质为“股权转让款及利息”；账龄“1年以内、1 - 2年”；坏账准备期末余额15,104,850.00元。  
2. **北京光辉必成投资有限公司**：期末余额18,000,000.00元，占比7.75%；款项性质为“投资意向金”；账龄“1年以内”；坏账准备期末余额900,000.00元。  
3. **北京安佳康医疗科技有限公司**：期末余额15,650,000.00元，占比6.74%；款项性质为“单位往来款”；账龄“1年以内”；坏账准备期末余额782,500.00元。  
4. **汉中市国土资源局经济开发区分局**：期末余额10,000,000.00元，占比4.30%；款项性质为“单位往来款”；账龄“1年以内”；坏账准备期末余额500,000.00元。  
5. **陕西超凡晶品建筑工程有限公司**：期末余额5,083,575.20元，占比2.19%；款项性质为“单位往来款”；账龄“1年以内、1 - 2年”；坏账准备期末余额458,001.68元。  

最后一行“合计”：期末余额202,014,275.20元，占比86.95%；坏账准备期末余额17,745,351.68元（“款项的性质”“账龄”列无数据）。  

表格通过列示各单位的其他应收款期末余额、占比、款项性质、账龄及坏账准备，清晰呈现了应收款的构成与坏账准备情况。
